# Week 3, day 3 (afternoon) — Worksheet 05: Validating the load   (runs locally)

`4_Validate_stage_tables.sql` validates the ingestion like this:

```sql
SELECT BATCH_ID, INSERTED_AT, COUNT(*) AS row_count
FROM STG_Sales
GROUP BY BATCH_ID, INSERTED_AT;
```

Row counts per batch. That is a real check — it catches an aborted load and a
double-load — and it is where most tutorials stop.

This worksheet is the rest of it. Everything below runs against a staging table
that already passed that check: 100,000 rows, one batch, no errors. The question
is what you can still find, and what it would have cost to find it later.

The final question is the one to think hardest about.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 05 — Validating the load. Run this once.
import pandas as pd

STG_SALES_COLS = [
    "TRANS_ID", "PROD_KEY", "STORE_KEY", "TRANS_DT", "TRANS_TIME",
    "PRIORITY", "SALES_QTY", "SALES_PRICE", "SALES_AMT",
    "DISCOUNT", "SALES_COST", "SALES_MGRN", "SHIP_MODE", "SHIP_COST",
]
STG_PRODUCT_COLS = [
    "PROD_KEY", "PROD_NAME", "VOL", "WGT", "BRAND_NAME",
    "STATUS_CODE", "STATUS_CODE_NAME", "CATEGORY_KEY", "CATEGORY_NAME",
    "SUBCATEGORY_KEY", "SUBCATEGORY_NAME",
]


def load_stg_sales():
    """STG_Sales exactly as 3_Stage_tables.sql builds it, quotes stripped."""
    df = pd.read_csv("data/sales_2013_01_01.csv", header=None, skiprows=1,
                     names=STG_SALES_COLS)
    for col in ("PRIORITY", "SHIP_MODE"):
        df[col] = df[col].str.strip('"')
    df["TRANS_DT"] = pd.to_datetime(df["TRANS_DT"], format="%m/%d/%Y")
    df["BATCH_ID"] = "sales_2013_01_01.csv"
    return df


def load_stg_products():
    df = pd.read_csv("data/products_2013_01_01.csv", header=None, skiprows=1,
                     names=STG_PRODUCT_COLS)
    df["BATCH_ID"] = "products_2013_01_01.csv"
    return df


sales = load_stg_sales()
products = load_stg_products()
print("STG_Sales:   ", sales.shape)
print("STG_Products:", products.shape)

PART A — the check the lab runs

### Question 1

Reproduce `4_Validate_stage_tables.sql` for both tables: group by `BATCH_ID` and count rows. Print both results.

In [ ]:
############################
## Your Code Here
############################

### Question 2

Show what that check is for. Simulate re-running the COPY by concatenating `sales` with a second `load_stg_sales()`, then run the same group-and-count. Print the count before and after.
> **NOTE:** `COPY INTO` skips files it has already loaded — for 64 days, tracked per table. `FORCE = TRUE` or a stage reset defeats that.

In [ ]:
############################
## Your Code Here
############################

PART B — the checks it does not run

### Question 3

Write a small completeness report over `sales`: for every column, the number of nulls and the number of distinct values. Print it sorted by distinct count.

In [ ]:
############################
## Your Code Here
############################

### Question 4

`STORE_KEY` has very few distinct values. Print the row count per store, sorted descending, and the share of total rows held by the largest one.

In [ ]:
############################
## Your Code Here
############################

### Question 5

Check the date column for gaps. Build a full daily range from the minimum to the maximum `TRANS_DT` and print the total days in the range, the days present, and the days missing.

In [ ]:
############################
## Your Code Here
############################

### Question 6

Look at the missing days. Print how many of them fall on each weekday name, using `missing.day_name()`.

In [ ]:
############################
## Your Code Here
############################

PART C — numbers that pass every null check

### Question 7

Print `describe()` for `SALES_QTY`, `SALES_AMT`, `SALES_COST` and `SALES_MGRN`, rounded to 2dp. Note the minimum of each.

In [ ]:
############################
## Your Code Here
############################

### Question 8

Count the rows where `SALES_MGRN` is negative, and their share of the total. Then check whether the margin is at least self-consistent: how many rows satisfy `SALES_MGRN == SALES_AMT - SALES_COST` to 2dp?

In [ ]:
############################
## Your Code Here
############################

PART D — what a join does with a duplicated key

### Question 9

Join `sales` to `products` on `PROD_KEY` with an inner merge. Print the row count before and after, and the total `SALES_AMT` before and after.
> **NOTE:** Worksheet 03 Q7 found three duplicate `PROD_KEY` rows in a 1,215-row table. This is what they cost.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Finally, read STEP 3 of `4_Validate_stage_tables.sql` — `DROP TABLE CORE.DIM_CALENDAR;` — then search every file in `snowflake-scripts/` and `labs/` for `DIM_CALENDAR`. Print each hit with its file and line, and count how many of them CREATE the table. Then say what happens when a student runs all four scripts in order.
> **NOTE:** read the scripts; this question is not about pandas. `glob.glob("snowflake-scripts/*.sql") + glob.glob("labs/*.html")`.

In [ ]:
############################
## Your Code Here
############################